# Árvores AVL — Tutorial

## Objetivos

Ao final deste tutorial você será capaz de:

- Representar nós e árvores AVL em Python.
- Calcular alturas e o fator de balanceamento.
- Implementar rotações simples (esquerda e direita).
- Combinar rotações para os casos duplos (Esq-Dir e Dir-Esq).
- Inserir chaves preservando a propriedade AVL.
- Visualizar a árvore para conferir o balanceamento.


## 1. Estrutura de um nó AVL

Cada nó guarda a chave, ponteiros para filhos e a altura da sub-árvore enraizada nele. Manter a altura como atributo evita recalculá-la a cada operação.

In [ ]:
# Exemplo: nó AVL básico
class AVLNode:
    def __init__(self, key):
        self.key = key       # chave armazenada
        self.left = None     # filho esquerdo
        self.right = None    # filho direito
        self.height = 1      # folha tem altura 1

def height(n):
    return n.height if n else 0

def balance_factor(n):
    return height(n.left) - height(n.right) if n else 0

raiz = AVLNode(30)
print('altura raiz =', height(raiz), '| bf =', balance_factor(raiz))

In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

def update_height(n):
    altura_esquerda = n.left.height if n.left else 0
    altura_direita = n.right.height if n.right else 0
    n.height = 1 + max(altura_esquerda, altura_direita)

n = AVLNode(10)
n.left = AVLNode(5)
n.right = AVLNode(15)
n.right.right = AVLNode(20)
n.right.height = 2
update_height(n)
assert n.height == 3, f'esperava 3, obtive {n.height}'
print('OK')

## 2. Rotações simples

A rotação é uma operação local de três ponteiros que mantém a ordem da BST. Existem duas variantes simétricas: à esquerda e à direita.

In [ ]:
# Exemplo: rotação à direita
def rotate_right(z):
    y = z.left           # novo topo
    T3 = y.right         # sub-árvore que muda de pai
    y.right = z          # z desce à direita de y
    z.left = T3          # z herda T3
    z.height = 1 + max(height(z.left), height(z.right))
    y.height = 1 + max(height(y.left), height(y.right))
    return y

# Monta cenário desbalanceado: 30 - 20 - 10 (caso Esq-Esq)
z = AVLNode(30); z.left = AVLNode(20); z.left.left = AVLNode(10)
z.left.height = 2; z.height = 3
novo = rotate_right(z)
print('nova raiz =', novo.key, '| esq =', novo.left.key, '| dir =', novo.right.key)

In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

def update_height(n):
    altura_esquerda = n.left.height if n.left else 0
    altura_direita = n.right.height if n.right else 0
    n.height = 1 + max(altura_esquerda, altura_direita)

def rotate_left(z):
    y = z.right
    t2 = y.left

    y.left = z
    z.right = t2

    update_height(z)
    update_height(y)

    return y

z = AVLNode(10)
z.right = AVLNode(20)
z.right.right = AVLNode(30)
z.right.height = 2
z.height = 3
r = rotate_left(z)
assert r.key == 20 and r.left.key == 10 and r.right.key == 30
print('OK')

## 3. Inserção com rebalanceamento

Após a inserção recursiva padrão de BST, voltamos pela pilha de chamadas atualizando alturas e aplicando rotações conforme o fator de balanceamento e o caminho do novo nó.

In [ ]:
# Exemplo: inserção AVL completa
def insert(node, key):
    if node is None:
        return AVLNode(key)
    if key < node.key:
        node.left = insert(node.left, key)
    elif key > node.key:
        node.right = insert(node.right, key)
    else:
        return node  # chaves duplicadas: ignora

    node.height = 1 + max(height(node.left), height(node.right))
    bf = balance_factor(node)

    # Caso Esq-Esq
    if bf > 1 and key < node.left.key:
        return rotate_right(node)
    # Caso Dir-Dir
    if bf < -1 and key > node.right.key:
        return rotate_left(node)
    # Caso Esq-Dir
    if bf > 1 and key > node.left.key:
        node.left = rotate_left(node.left)
        return rotate_right(node)
    # Caso Dir-Esq
    if bf < -1 and key < node.right.key:
        node.right = rotate_right(node.right)
        return rotate_left(node)
    return node

raiz = None
for k in [10, 20, 30, 40, 50, 25]:
    raiz = insert(raiz, k)
print('raiz após inserções =', raiz.key, '| altura =', raiz.height)

In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

def get_height(n):
    return n.height if n else 0

def update_height(n):
    n.height = 1 + max(get_height(n.left), get_height(n.right))

def get_balance(n):
    return get_height(n.left) - get_height(n.right) if n else 0

def rotate_right(y):
    x = y.left
    t2 = x.right
    x.right = y
    y.left = t2
    update_height(y)
    update_height(x)
    return x

def rotate_left(x):
    y = x.right
    t2 = y.left
    y.left = x
    x.right = t2
    update_height(x)
    update_height(y)
    return y

def insert(node, key):
    if not node:
        return AVLNode(key)

    if key < node.key:
        node.left = insert(node.left, key)
    elif key > node.key:
        node.right = insert(node.right, key)
    else:
        return node

    update_height(node)
    balance = get_balance(node)

    if balance > 1 and key < node.left.key:
        return rotate_right(node)

    if balance < -1 and key > node.right.key:
        return rotate_left(node)

    if balance > 1 and key > node.left.key:
        node.left = rotate_left(node.left)
        return rotate_right(node)

    if balance < -1 and key < node.right.key:
        node.right = rotate_right(node.right)
        return rotate_left(node)

    return node

def is_avl(node):
    if not node:
        return True

    balance = get_balance(node)
    if abs(balance) > 1:
        return False

    return is_avl(node.left) and is_avl(node.right)

r = None
for k in [50, 30, 70, 20, 40, 60, 80, 10]:
    r = insert(r, k)
assert is_avl(r), 'a árvore deveria ser AVL após inserções com rebalanceamento'
print('OK')

## 4. Visualização

Uma visualização rápida ajuda a perceber o equilíbrio. Usamos `matplotlib` para desenhar a árvore por níveis.

In [ ]:
# Visualização: árvore AVL
import matplotlib.pyplot as plt

def plot_tree(node, x=0.0, y=0.0, dx=1.0, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4)); ax.set_axis_off()
    if node is None:
        return ax
    ax.text(x, y, str(node.key), ha='center', va='center',
            bbox=dict(boxstyle='circle', fc='#cfe2ff', ec='#034ea2'))
    if node.left:
        ax.plot([x, x - dx], [y - 0.1, y - 1 + 0.1], color='#034ea2')
        plot_tree(node.left, x - dx, y - 1, dx / 1.8, ax)
    if node.right:
        ax.plot([x, x + dx], [y - 0.1, y - 1 + 0.1], color='#034ea2')
        plot_tree(node.right, x + dx, y - 1, dx / 1.8, ax)
    return ax

r = None
for k in [50, 30, 70, 20, 40, 60, 80, 10, 25, 5]:
    r = insert(r, k)
plot_tree(r)
plt.show()

## Desafio Final

Implemente a operação `delete(node, key)` que remove uma chave da AVL e rebalanceia a árvore se necessário. Valide com a função `is_avl` do Exercício 3 após várias remoções aleatórias.


In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

def get_height(n):
    return n.height if n else 0

def update_height(n):
    n.height = 1 + max(get_height(n.left), get_height(n.right))

def get_balance(n):
    return get_height(n.left) - get_height(n.right) if n else 0

def rotate_right(y):
    x = y.left
    t2 = x.right
    x.right = y
    y.left = t2
    update_height(y)
    update_height(x)
    return x

def rotate_left(x):
    y = x.right
    t2 = y.left
    y.left = x
    x.right = t2
    update_height(x)
    update_height(y)
    return y

def get_min_value_node(node):
    current = node
    while current.left is not None:
        current = current.left
    return current

def is_avl(node):
    if not node:
        return True
    balance = get_balance(node)
    if abs(balance) > 1:
        return False
    return is_avl(node.left) and is_avl(node.right)

def insert(node, key):
    if not node:
        return AVLNode(key)
    if key < node.key:
        node.left = insert(node.left, key)
    elif key > node.key:
        node.right = insert(node.right, key)
    else:
        return node

    update_height(node)
    balance = get_balance(node)

    if balance > 1 and key < node.left.key:
        return rotate_right(node)
    if balance < -1 and key > node.right.key:
        return rotate_left(node)
    if balance > 1 and key > node.left.key:
        node.left = rotate_left(node.left)
        return rotate_right(node)
    if balance < -1 and key < node.right.key:
        node.right = rotate_right(node.right)
        return rotate_left(node)

    return node

def delete(node, key):
    if not node:
        return node

    if key < node.key:
        node.left = delete(node.left, key)
    elif key > node.key:
        node.right = delete(node.right, key)
    else:
        if node.left is None:
            temp = node.right
            node = None
            return temp
        elif node.right is None:
            temp = node.left
            node = None
            return temp

        temp = get_min_value_node(node.right)
        node.key = temp.key
        node.right = delete(node.right, temp.key)

    if node is None:
        return node

    update_height(node)
    balance = get_balance(node)

    if balance > 1 and get_balance(node.left) >= 0:
        return rotate_right(node)
    if balance > 1 and get_balance(node.left) < 0:
        node.left = rotate_left(node.left)
        return rotate_right(node)
    if balance < -1 and get_balance(node.right) <= 0:
        return rotate_left(node)
    if balance < -1 and get_balance(node.right) > 0:
        node.right = rotate_right(node.right)
        return rotate_left(node)

    return node


import random

r = None
chaves = [50, 30, 70, 20, 40, 60, 80, 10, 90, 5, 15, 25, 35, 45, 55, 65, 75, 85, 95]
for k in chaves:
    r = insert(r, k)

random.shuffle(chaves)
for k in chaves:
    r = delete(r, k)
    assert is_avl(r)

## Referências

Veja o arquivo `../referencias.bib` para a lista completa.
